# User Story: Fine-tuning Deepfake Detection Model

## Overview
This notebook provides a complete pipeline for fine-tuning a pre-trained Xception model on deepfake detection using face crops extracted from video data.

## Workflow

### Step 1: Data Splitting (Prerequisite)
**Before using this notebook**, you must first split your video data using the `split_chunks_00_into_train_val_test_john.ipynb` notebook:
- Use the **exact same splitting method** (80/10/10 train/val/test ratio with random_state=42)
- The split notebook organizes videos into the following structure:
  ```
  organized/
    train/
      real_videos/*.mp4
      fake_videos/*.mp4
    val/
      real_videos/*.mp4
      fake_videos/*.mp4
    test/
      real_videos/*.mp4
      fake_videos/*.mp4
  ```

### Step 2: Update Configuration
After splitting your data, update the folder paths in the **User Configuration** section (Cell 0) to point to your sample data:
- Change `data_path` to your organized data directory
- Update `frames_base_dir`, `crops_base_dir`, and `run_base_dir` as needed

### Step 3: Run Fine-tuning Pipeline
This notebook will:
1. Extract frames from videos (3 fps)
2. Detect and crop faces from frames using MediaPipe
3. Fine-tune Xception model in two phases:
   - **Phase 1**: Freeze backbone, train classifier head only
   - **Phase 2**: Unfreeze all layers, fine-tune entire model
4. Track and visualize:
   - Validation loss over epochs
   - Confusion matrix for validation set
5. Evaluate on test set with metrics and confusion matrix

## Expected Outputs
- Trained model checkpoints (best and last)
- Validation loss graph showing training progress
- Confusion matrix visualization
- Test set evaluation metrics (Accuracy, F1, AUC, AP)


In [ ]:
# --- User Configuration ---
# Specify the base directory where your organized video data is located.
# This directory should contain 'train', 'val', and 'test' subdirectories,
# each with 'real_videos' and 'fake_videos' folders containing .mp4 files.
data_path = '/content/drive/MyDrive/DeepfakeData/CompleteData/organized'

# Specify the base directory where extracted frames will be saved.
# This directory will mirror the structure of data_path.
frames_base_dir = '/content/drive/MyDrive/DeepfakeData/CompleteData/organized/frames'

# Specify the base directory where cropped faces will be saved.
# This directory will mirror the structure of data_path.
crops_base_dir = '/content/drive/MyDrive/DeepfakeData/CompleteData/organized/crops'

# Specify the base directory where training runs and checkpoints will be saved.
run_base_dir = '/content/drive/MyDrive/DeepfakeData/CompleteData/organized/runs'

# --- End User Configuration ---

In [ ]:
import os

destination_base = '/content/drive/MyDrive/DeepfakeData/CompleteData/organized/frames'
subdirectories = [
    'train/real_videos',
    'train/fake_videos',
    'val/real_videos',
    'val/fake_videos',
    'test/real_videos',
    'test/fake_videos'
]

for subdir in subdirectories:
    full_path = os.path.join(destination_base, subdir)
    os.makedirs(full_path, exist_ok=True)

print("Directories created successfully.")

Directories created successfully.


In [ ]:
import os

video_files = []
main_subdirs = ['train', 'val', 'test']
inner_subdirs = ['real_videos', 'fake_videos']

for main_subdir in main_subdirs:
    main_subdir_path = os.path.join(data_path, main_subdir)
    if os.path.isdir(main_subdir_path):
        for inner_subdir in inner_subdirs:
            inner_subdir_path = os.path.join(main_subdir_path, inner_subdir)
            if os.path.isdir(inner_subdir_path):
                for filename in os.listdir(inner_subdir_path):
                    if filename.endswith('.mp4'):
                        video_files.append(os.path.join(inner_subdir_path, filename))

print(f"Found {len(video_files)} video files.")

Found 1408 video files.


In [ ]:
import subprocess
import os

for video_file in video_files:
    relative_path = os.path.relpath(video_file, data_path)
    # The output directory structure will mirror the input structure within frames_base_dir
    output_dir = os.path.join(frames_base_dir, os.path.dirname(relative_path), os.path.splitext(os.path.basename(video_file))[0])

    # Check if the output directory already exists and is not empty
    if os.path.isdir(output_dir) and len(os.listdir(output_dir)) > 0:
        print(f"Frames for {video_file} already seem to be extracted. Skipping.")
        continue # Skip to the next video file

    os.makedirs(output_dir, exist_ok=True)
    output_frame_pattern = os.path.join(output_dir, '%04d.png')

    ffmpeg_command = [
        'ffmpeg',
        '-i', video_file,
        '-vf', 'fps=3',
        output_frame_pattern
    ]

    try:
        # Added capture_output=True and text=True to capture stderr and stdout
        result = subprocess.run(ffmpeg_command, check=True, capture_output=True, text=True)
        print(f"Successfully extracted frames from {video_file}")
        # Optionally print stdout/stderr for debugging
        # print(f"Stdout: {result.stdout}")
        # print(f"Stderr: {result.stderr}")
    except subprocess.CalledProcessError as e:
        print(f"Error extracting frames from {video_file}: {e}")
        print(f"Stderr: {e.stderr}")
    except FileNotFoundError:
        print("Error: ffmpeg command not found. Make sure ffmpeg is installed and in your PATH.")
    except Exception as e:
        print(f"An unexpected error occurred while processing {video_file}: {e}")

Frames for /content/drive/MyDrive/DeepfakeData/CompleteData/organized/train/real_videos/wbuajbdcfs.mp4 already seem to be extracted. Skipping.
Frames for /content/drive/MyDrive/DeepfakeData/CompleteData/organized/train/real_videos/iorbtaarte.mp4 already seem to be extracted. Skipping.
Frames for /content/drive/MyDrive/DeepfakeData/CompleteData/organized/train/real_videos/iksxzpqxzi.mp4 already seem to be extracted. Skipping.
Frames for /content/drive/MyDrive/DeepfakeData/CompleteData/organized/train/real_videos/yiykshcbaz.mp4 already seem to be extracted. Skipping.
Frames for /content/drive/MyDrive/DeepfakeData/CompleteData/organized/train/real_videos/uazbhwyysx.mp4 already seem to be extracted. Skipping.
Frames for /content/drive/MyDrive/DeepfakeData/CompleteData/organized/train/real_videos/cekarydqba.mp4 already seem to be extracted. Skipping.
Frames for /content/drive/MyDrive/DeepfakeData/CompleteData/organized/train/real_videos/fglewmddcn.mp4 already seem to be extracted. Skipping.

In [ ]:
%pip install mediapipe

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.0 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of opencv-contrib-python to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.6/35.6 MB 73.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 102.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.9/294.9 kB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 MB 38.3 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.5
    Uninstalling protobuf-5.29.5:
      Successfully uninstalled protobuf-5.29.5
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
  Attempting uninstall: opencv-contrib-python
    Found existing installation: opencv-contrib-python 4.12.0.88
    Uninstalling open

In [ ]:
#!/usr/bin/env python3
"""
crop_faces.py — Detect, 1.3x enlarge, and save 299x299 face crops from frame folders.
Useful to prepare ImageFolder datasets for training.

Input frames dir structure:
frames/
  videoA/frame_00001.jpg
  videoA/frame_00002.jpg
  ...
  videoB/frame_00001.jpg
  ...

Usage:
Run the crop_faces_from_video_folder function with appropriate arguments.
"""

import argparse, os
from pathlib import Path
import cv2
import mediapipe as mp

def enlarge_box(x1,y1,x2,y2, scale, W, H):
    cx, cy = (x1+x2)/2.0, (y1+y2)/2.0
    w, h = (x2-x1)*scale, (y2-y1)*scale
    nx1, ny1 = max(0, int(cx - w/2)), max(0, int(cy - h/2))
    nx2, ny2 = min(W-1, int(cx + w/2)), min(H-1, int(cy + h/2))
    return nx1, ny1, nx2, ny2

def detect_largest_face_mediapipe(img, face_detector):
    # Convert the image to RGB as mediapipe requires RGB input
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    results = face_detector.process(img_rgb)

    if not results.detections:
        return None

    largest_face = None
    largest_area = 0

    for detection in results.detections:
        bbox = detection.location_data.relative_bounding_box
        ih, iw, _ = img.shape
        x, y, w, h = int(bbox.xmin * iw), int(bbox.ymin * ih), int(bbox.width * iw), int(bbox.height * ih)
        area = w * h
        if area > largest_area:
            largest_area = area
            largest_face = (x, y, x+w, y+h)

    return largest_face


def crop_faces_from_video_folder(frames_dir, out_dir, enlarge_scale=1.3, jpeg_quality=95):
    """
    Detect, enlarge, and save face crops from frame folders.

    Args:
        frames_dir (str): Directory with per-video subfolders of frames.
        out_dir (str): Where to write 299x299 crops.
        enlarge_scale (float, optional): Scale to enlarge the face bounding box. Defaults to 1.3.
        jpeg_quality (int, optional): JPEG quality for saved images. Defaults to 95.
    """
    mp_face_detection = mp.solutions.face_detection
    mp_drawing = mp.solutions.drawing_utils

    frames_root = Path(frames_dir)
    out_root = Path(out_dir)
    out_root.mkdir(parents=True, exist_ok=True)

    videos = sorted([p for p in frames_root.iterdir() if p.is_dir()])
    count = 0

    # Initialize Mediapipe Face Detection
    with mp_face_detection.FaceDetection(
        model_selection=1, min_detection_confidence=0.5) as face_detector:

        for vdir in videos:
            # Change glob pattern to look for .png files
            for fp in sorted(vdir.glob("*.png")):
                out_fp = out_root / f"{vdir.name}_{fp.stem}.jpg" # Save as .jpg as requested

                # Check if cropped file already exists
                if out_fp.exists():
                    # print(f"Crop for {fp} already exists. Skipping.") # Optional: uncomment for verbose skipping
                    continue

                img = cv2.imread(str(fp))
                if img is None:
                    continue
                H, W = img.shape[:2]
                box = detect_largest_face_mediapipe(img, face_detector)
                if box is None:
                    continue
                x1,y1,x2,y2 = box
                x1,y1,x2,y2 = enlarge_box(x1,y1,x2,y2, enlarge_scale, W, H)
                crop = img[y1:y2, x1:x2]
                crop = cv2.resize(crop, (299,299), interpolation=cv2.INTER_AREA)

                cv2.imwrite(str(out_fp), crop, [int(cv2.IMWRITE_JPEG_QUALITY), jpeg_quality])
                count += 1

    print(f"Saved {count} new crops to {out_root}")

# Example usage (you can call this function in a separate cell):
# frames_dir = '/content/drive/MyDrive/DeepfakeData/SampleData/frames' # Replace with your frames directory
# out_dir = '/content/drive/MyDrive/DeepfakeData/SampleData/crops/train/real' # Replace with your desired output directory
# crop_faces_from_video_folder(frames_dir, out_dir)

In [ ]:
import os

subdirectories = [
    'train/real_videos',
    'train/fake_videos',
    'val/real_videos',
    'val/fake_videos',
    'test/real_videos',
    'test/fake_videos'
]

for subdir in subdirectories:
    frames_subdir = os.path.join(frames_base_dir, subdir)
    print(f"Checking directory: {frames_subdir}")
    if os.path.isdir(frames_subdir):
        print("Directory exists.")
        # Check for files within the directory (limit to first few for brevity)
        files_in_dir = os.listdir(frames_subdir)
        if files_in_dir:
            print(f"Found {len(files_in_dir)} items in directory. First 5: {files_in_dir[:5]}")
            # Further check if any expected image files exist within the subdirectories (video folders)
            video_subdirs = [d for d in os.listdir(frames_subdir) if os.path.isdir(os.path.join(frames_subdir, d))]
            if video_subdirs:
                print(f"Found {len(video_subdirs)} video subdirectories. Checking the first one...")
                first_video_subdir = video_subdirs[0]
                first_video_subdir_path = os.path.join(frames_subdir, first_video_subdir)
                frames_in_video_subdir = [f for f in os.listdir(first_video_subdir_path) if f.endswith('.jpg') or f.endswith('.png')]
                if frames_in_video_subdir:
                    print(f"Found {len(frames_in_video_subdir)} image files in {first_video_subdir}. First 5: {frames_in_video_subdir[:5]}")
                else:
                    print(f"No image files found in {first_video_subdir}.")
            else:
                print("No video subdirectories found within this directory.")
        else:
            print("Directory is empty.")
    else:
        print("Directory does not exist.")
    print("-" * 30)

Checking directory: /content/drive/MyDrive/DeepfakeData/CompleteData/organized/frames/train/real_videos
Directory exists.
Found 142 items in directory. First 5: ['wbuajbdcfs', 'iorbtaarte', 'iksxzpqxzi', 'yiykshcbaz', 'uazbhwyysx']
Found 142 video subdirectories. Checking the first one...
Found 30 image files in wbuajbdcfs. First 5: ['0001.png', '0002.png', '0003.png', '0004.png', '0005.png']
------------------------------
Checking directory: /content/drive/MyDrive/DeepfakeData/CompleteData/organized/frames/train/fake_videos
Directory exists.
Found 998 items in directory. First 5: ['owxbbpjpch', 'scpglaliyh', 'inzrlbgtul', 'sphirandia', 'ztwlbdwyni']
Found 998 video subdirectories. Checking the first one...
Found 30 image files in owxbbpjpch. First 5: ['0001.png', '0002.png', '0003.png', '0004.png', '0005.png']
------------------------------
Checking directory: /content/drive/MyDrive/DeepfakeData/CompleteData/organized/frames/val/real_videos
Directory exists.
Found 9 items in directory

In [ ]:
import os

# Define the base directories
# Using the variables defined at the top of the notebook
# Define the subdirectories for frames and crops
subdirectories = [
    'train/real_videos',
    'train/fake_videos',
    'val/real_videos',
    'val/fake_videos',
    'test/real_videos',
    'test/fake_videos'
]

# Process each subdirectory
for subdir in subdirectories:
    frames_subdir = os.path.join(frames_base_dir, subdir)
    crops_subdir = os.path.join(crops_base_dir, subdir)

    # Ensure the output directory exists
    os.makedirs(crops_subdir, exist_ok=True)

    print(f"Processing frames from: {frames_subdir}")
    print(f"Saving crops to: {crops_subdir}")

    # Call the crop_faces_from_video_folder function
    crop_faces_from_video_folder(frames_subdir, crops_subdir)
    print("-" * 30)

print("Face cropping process completed for all subdirectories.")

Processing frames from: /content/drive/MyDrive/DeepfakeData/CompleteData/organized/frames/train/real_videos
Saving crops to: /content/drive/MyDrive/DeepfakeData/CompleteData/organized/crops/train/real_videos
Saved 0 new crops to /content/drive/MyDrive/DeepfakeData/CompleteData/organized/crops/train/real_videos
------------------------------
Processing frames from: /content/drive/MyDrive/DeepfakeData/CompleteData/organized/frames/train/fake_videos
Saving crops to: /content/drive/MyDrive/DeepfakeData/CompleteData/organized/crops/train/fake_videos
Saved 0 new crops to /content/drive/MyDrive/DeepfakeData/CompleteData/organized/crops/train/fake_videos
------------------------------
Processing frames from: /content/drive/MyDrive/DeepfakeData/CompleteData/organized/frames/val/real_videos
Saving crops to: /content/drive/MyDrive/DeepfakeData/CompleteData/organized/crops/val/real_videos
Saved 0 new crops to /content/drive/MyDrive/DeepfakeData/CompleteData/organized/crops/val/real_videos
--------

In [ ]:
import os

subdirectories = [
    'train/real_videos',
    'train/fake_videos',
    'val/real_videos',
    'val/fake_videos',
    'test/real_videos',
    'test/fake_videos'
]

print("Checking if crop directories contain data:")

for subdir in subdirectories:
    crops_subdir = os.path.join(crops_base_dir, subdir)
    print(f"Checking directory: {crops_subdir}")
    if os.path.isdir(crops_subdir):
        files_in_dir = os.listdir(crops_subdir)
        if files_in_dir:
            print(f"Directory exists and contains {len(files_in_dir)} files. First 5: {files_in_dir[:5]}")
        else:
            print("Directory exists but is empty.")
    else:
        print("Directory does not exist.")
    print("-" * 30)

Checking if crop directories contain data:
Checking directory: /content/drive/MyDrive/DeepfakeData/CompleteData/organized/crops/train/real_videos
Directory exists and contains 4175 files. First 5: ['ucthmsajay_0017.jpg', 'ucthmsajay_0018.jpg', 'ucthmsajay_0019.jpg', 'ucthmsajay_0020.jpg', 'ucthmsajay_0021.jpg']
------------------------------
Checking directory: /content/drive/MyDrive/DeepfakeData/CompleteData/organized/crops/train/fake_videos
Directory exists and contains 29804 files. First 5: ['zbvlctwcqr_0016.jpg', 'zbvlctwcqr_0017.jpg', 'zbvlctwcqr_0018.jpg', 'zbvlctwcqr_0019.jpg', 'zbvlctwcqr_0020.jpg']
------------------------------
Checking directory: /content/drive/MyDrive/DeepfakeData/CompleteData/organized/crops/val/real_videos
Directory exists and contains 267 files. First 5: ['doniqevxeg_0001.jpg', 'doniqevxeg_0002.jpg', 'doniqevxeg_0003.jpg', 'doniqevxeg_0004.jpg', 'doniqevxeg_0005.jpg']
------------------------------
Checking directory: /content/drive/MyDrive/DeepfakeData/

In [ ]:
%pip install timm==0.6.13
%pip install seaborn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 549.1/549.1 kB 12.0 MB/s eta 0:00:00
  Attempting uninstall: timm
    Found existing installation: timm 1.0.20
    Uninstalling timm-1.0.20:
      Successfully uninstalled timm-1.0.20


In [ ]:
!pip install torch_xla

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.9/106.9 MB 3.4 MB/s eta 0:00:00


In [ ]:
!pip install --upgrade torchvision

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 48.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.7/899.7 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 140.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 67.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 77.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 41.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.5/267.5 MB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 288.2/288.2 MB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.3/322.3 MB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.

In [ ]:
#!/usr/bin/env python3
"""
train_xception.py — Fine-tune XceptionNet (via timm) on face crops (real/fake)
Dataset layout (ImageFolder style):
crops/
  train/real/*.jpg
  train/fake/*.jpg
  val/real/*.jpg
  val/fake/*.jpg

Usage:
Run the train_model function with appropriate arguments.
"""

import argparse, os, random, time, math
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

try:
    import timm
except Exception as e:
    raise SystemExit("Please install timm: pip install timm==1.*") from e

from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, average_precision_score, confusion_matrix

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

In [ ]:
def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def build_loaders(data_dir, img_size=299, batch_size=64, num_workers=8):
    train_tf = transforms.Compose([
        transforms.RandomResizedCrop(img_size, scale=(0.9, 1.0)),
        transforms.RandomApply([transforms.GaussianBlur(3)], p=0.2),
        transforms.ColorJitter(0.1,0.1,0.1,0.05),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])
    val_tf = transforms.Compose([
        transforms.Resize(img_size),
        transforms.CenterCrop(img_size),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])
    train_ds = datasets.ImageFolder(os.path.join(data_dir, "train"), transform=train_tf)
    val_ds   = datasets.ImageFolder(os.path.join(data_dir, "val"),   transform=val_tf)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                              num_workers=num_workers, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False,
                              num_workers=num_workers, pin_memory=True)
    return train_loader, val_loader, train_ds.classes

In [ ]:
def evaluate_frame_level(model, loader, device):
    model.eval()
    all_logits, all_targets = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            logits = model(x)
            all_logits.append(logits.cpu())
            all_targets.append(y)
    logits = torch.cat(all_logits, dim=0)
    targets = torch.cat(all_targets, dim=0).numpy()
    probs = torch.softmax(logits, dim=1).numpy()[:,1]
    preds = (probs >= 0.5).astype(np.int64)
    metrics = {}
    metrics["acc"] = float(accuracy_score(targets, preds))
    metrics["f1"]  = float(f1_score(targets, preds))
    try:
        metrics["auc"] = float(roc_auc_score(targets, probs))
        metrics["ap_fake"] = float(average_precision_score(targets, probs))
    except Exception:
        metrics["auc"] = float("nan")
        metrics["ap_fake"] = float("nan")
    
    # Calculate confusion matrix
    cm = confusion_matrix(targets, preds)
    
    return metrics, cm

In [ ]:
def save_checkpoint(state, is_best, out_dir, name="xception"):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    ckpt = out_dir / f"{name}_last.pt"
    torch.save(state, ckpt)
    if is_best:
        best = out_dir / f"{name}_best.pt"
        torch.save(state, best)

In [ ]:
def evaluate_with_loss(model, loader, criterion, device):
    """Evaluate model and return metrics, confusion matrix, and validation loss"""
    model.eval()
    all_logits, all_targets = [], []
    val_losses = []
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = criterion(logits, y)
            val_losses.append(loss.item())
            all_logits.append(logits.cpu())
            all_targets.append(y)
    logits = torch.cat(all_logits, dim=0)
    targets = torch.cat(all_targets, dim=0).numpy()
    probs = torch.softmax(logits, dim=1).numpy()[:,1]
    preds = (probs >= 0.5).astype(np.int64)
    
    val_loss = float(np.mean(val_losses))
    
    metrics = {}
    metrics["acc"] = float(accuracy_score(targets, preds))
    metrics["f1"]  = float(f1_score(targets, preds))
    try:
        metrics["auc"] = float(roc_auc_score(targets, probs))
        metrics["ap_fake"] = float(average_precision_score(targets, probs))
    except Exception:
        metrics["auc"] = float("nan")
        metrics["ap_fake"] = float("nan")
    
    # Calculate confusion matrix
    cm = confusion_matrix(targets, preds)
    
    return metrics, cm, val_loss

def train_model(data_dir, out_dir, epochs_head=3, epochs_full=15, batch_size=64, img_size=299, num_workers=8, seed=42, lr_head=3e-4, lr_full=1e-4, weight_decay=1e-4):
    seed_everything(seed)

    # Use CUDA device if available, otherwise CPU
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device}")

    train_loader, val_loader, classes = build_loaders(data_dir, img_size, batch_size, num_workers)
    print(f"Classes: {classes} (expect ['fake', 'real'] or similar two classes)")

    # Build model
    model = timm.create_model("xception", pretrained=True, num_classes=2)
    model = model.to(device)

    # Phase 1: freeze backbone, train head
    head_params = []
    for n, p in model.named_parameters():
        if "fc" in n or "classifier" in n:
            head_params.append(p)
            p.requires_grad = True
        else:
            p.requires_grad = False

    opt = torch.optim.AdamW(head_params, lr=lr_head, weight_decay=weight_decay)
    crit = nn.CrossEntropyLoss()

    best_val = -1.0
    
    # Track training history
    train_losses = []
    val_losses = []
    val_accs = []
    val_cms = []
    epochs_list = []

    def train_one_epoch(epoch, model, loader, optimizer, criterion, device):
        model.train()
        losses = []
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad(set_to_none=True)
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()
            losses.append(loss.item())
        return float(np.mean(losses))

    print("==> Phase 1: training the head")
    for epoch in range(1, epochs_head+1):
        tr_loss = train_one_epoch(epoch, model, train_loader, opt, crit, device)
        val_metrics, val_cm, val_loss = evaluate_with_loss(model, val_loader, crit, device)
        score = val_metrics["acc"]
        is_best = score > best_val
        if is_best: best_val = score
        
        # Store history
        train_losses.append(tr_loss)
        val_losses.append(val_loss)
        val_accs.append(score)
        val_cms.append(val_cm)
        epochs_list.append(epoch)
        
        save_checkpoint({
            "epoch": epoch,
            "model_state": model.state_dict(),
            "val_metrics": val_metrics,
            "val_loss": val_loss,
            "val_cm": val_cm.tolist(),
            "args": {
                "data_dir": data_dir, "out_dir": out_dir, "epochs_head": epochs_head,
                "epochs_full": epochs_full, "batch_size": batch_size, "img_size": img_size,
                "num_workers": num_workers, "seed": seed, "lr_head": lr_head,
                "lr_full": lr_full, "weight_decay": weight_decay
            },
        }, is_best, out_dir, name="xception")
        print(f"[Head {epoch}/{epochs_head}] train_loss={tr_loss:.4f} val_loss={val_loss:.4f} val={val_metrics}")
        print(f"Validation Confusion Matrix:\n{val_cm}")

    # Phase 2: unfreeze all, train
    for p in model.parameters():
        p.requires_grad = True
    opt = torch.optim.AdamW(model.parameters(), lr=lr_full, weight_decay=weight_decay)

    print("==> Phase 2: fine-tuning all layers")
    total_epochs = epochs_full
    for epoch in range(1, total_epochs+1):
        tr_loss = train_one_epoch(epoch, model, train_loader, opt, crit, device)
        val_metrics, val_cm, val_loss = evaluate_with_loss(model, val_loader, crit, device)
        score = val_metrics["acc"]
        is_best = score > best_val
        if is_best: best_val = score
        
        # Store history
        train_losses.append(tr_loss)
        val_losses.append(val_loss)
        val_accs.append(score)
        val_cms.append(val_cm)
        epochs_list.append(epochs_head + epoch)
        
        save_checkpoint({
            "epoch": epochs_head + epoch,
            "model_state": model.state_dict(),
            "val_metrics": val_metrics,
            "val_loss": val_loss,
            "val_cm": val_cm.tolist(),
            "args": {
                "data_dir": data_dir, "out_dir": out_dir, "epochs_head": epochs_head,
                "epochs_full": epochs_full, "batch_size": batch_size, "img_size": img_size,
                "num_workers": num_workers, "seed": seed, "lr_head": lr_head,
                "lr_full": lr_full, "weight_decay": weight_decay
            },
        }, is_best, out_dir, name="xception")
        print(f"[FT {epoch}/{total_epochs}] train_loss={tr_loss:.4f} val_loss={val_loss:.4f} val={val_metrics}")
        print(f"Validation Confusion Matrix:\n{val_cm}")

    print("Training complete. Best val acc:", best_val)
    
    # Return history for visualization
    return {
        'epochs': epochs_list,
        'train_losses': train_losses,
        'val_losses': val_losses,
        'val_accs': val_accs,
        'val_cms': val_cms,
        'best_val_cm': val_cms[np.argmax(val_accs)] if val_cms else None
    }
# Example usage (you can call this function in a separate cell):
data_dir = crops_base_dir
out_dir = f'{run_base_dir}/xception_dfdc' # Replace with your desired output directory for runs
training_history = train_model(data_dir, out_dir, batch_size=32) # Reduced batch size to mitigate OOM

Using device: cuda
Classes: ['fake_videos', 'real_videos'] (expect ['fake', 'real'] or similar two classes)
Downloading: "https://github.com/rwightman/pytorch-image-models/releases/download/v0.1-cadene/xception-43020ad28.pth" to /root/.cache/torch/hub/checkpoints/xception-43020ad28.pth
==> Phase 1: training the head
[Head 1/3] loss=0.2864  val={'acc': 0.932, 'f1': 0.0, 'auc': 0.7129749746917612, 'ap_fake': 0.13726350567721252}
[Head 2/3] loss=0.2400  val={'acc': 0.9325, 'f1': 0.0425531914893617, 'auc': 0.7506207917841782, 'ap_fake': 0.19467819341232157}
[Head 3/3] loss=0.2254  val={'acc': 0.931, 'f1': 0.08, 'auc': 0.760172206386806, 'ap_fake': 0.19761357171922744}
==> Phase 2: fine-tuning all layers
[FT 1/15] loss=0.0813  val={'acc': 0.98175, 'f1': 0.8665447897623401, 'auc': 0.9916284660247554, 'ap_fake': 0.9404713552989907}
[FT 2/15] loss=0.0384  val={'acc': 0.98625, 'f1': 0.8897795591182365, 'auc': 0.9808118903072205, 'ap_fake': 0.9374611388709205}
[FT 3/15] loss=0.0268  val={'acc': 

In [ ]:
# Visualize Validation Loss Over Training
import matplotlib.pyplot as plt

if 'training_history' in locals() and training_history:
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    epochs = training_history['epochs']
    train_losses = training_history['train_losses']
    val_losses = training_history['val_losses']
    val_accs = training_history['val_accs']
    
    # Plot 1: Training and Validation Loss
    axes[0].plot(epochs, train_losses, 'b-', label='Train Loss', linewidth=2)
    axes[0].plot(epochs, val_losses, 'r-', label='Validation Loss', linewidth=2)
    axes[0].set_xlabel('Epoch', fontsize=12)
    axes[0].set_ylabel('Loss', fontsize=12)
    axes[0].set_title('Training and Validation Loss Over Epochs', fontsize=14, fontweight='bold')
    axes[0].legend(fontsize=11)
    axes[0].grid(True, alpha=0.3)
    
    # Plot 2: Validation Accuracy
    axes[1].plot(epochs, val_accs, 'g-', label='Validation Accuracy', linewidth=2)
    axes[1].set_xlabel('Epoch', fontsize=12)
    axes[1].set_ylabel('Accuracy', fontsize=12)
    axes[1].set_title('Validation Accuracy Over Epochs', fontsize=14, fontweight='bold')
    axes[1].legend(fontsize=11)
    axes[1].grid(True, alpha=0.3)
    axes[1].set_ylim([0, 1])
    
    plt.tight_layout()
    plt.show()
    
    print(f"Best validation accuracy: {max(val_accs):.4f} at epoch {epochs[np.argmax(val_accs)]}")
    print(f"Lowest validation loss: {min(val_losses):.4f} at epoch {epochs[np.argmin(val_losses)]}")
else:
    print("Training history not found. Please run the training cell first.")


In [ ]:
# Visualize Confusion Matrix for Best Validation Model
import matplotlib.pyplot as plt
import seaborn as sns

if 'training_history' in locals() and training_history and training_history['best_val_cm'] is not None:
    best_cm = training_history['best_val_cm']
    
    plt.figure(figsize=(8, 6))
    # Note: Classes are ['fake_videos', 'real_videos'], so index 0=Fake, index 1=Real
    sns.heatmap(best_cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=['Fake', 'Real'], 
                yticklabels=['Fake', 'Real'],
                cbar_kws={'label': 'Count'})
    plt.title('Confusion Matrix - Best Validation Model', fontsize=14, fontweight='bold')
    plt.xlabel('Predicted Label', fontsize=12)
    plt.ylabel('True Label', fontsize=12)
    plt.tight_layout()
    plt.show()
    
    # Print confusion matrix details
    # best_cm structure: [[Fake->Fake, Fake->Real], [Real->Fake, Real->Real]]
    print("Confusion Matrix (Best Validation Model):")
    print(f"  True Positives (Fake->Fake): {best_cm[0][0]}")
    print(f"  False Negatives (Fake->Real): {best_cm[0][1]}")
    print(f"  False Positives (Real->Fake): {best_cm[1][0]}")
    print(f"  True Negatives (Real->Real): {best_cm[1][1]}")
    
    total = best_cm.sum()
    accuracy = (best_cm[0][0] + best_cm[1][1]) / total if total > 0 else 0
    precision = best_cm[0][0] / (best_cm[0][0] + best_cm[1][0]) if (best_cm[0][0] + best_cm[1][0]) > 0 else 0
    recall = best_cm[0][0] / (best_cm[0][0] + best_cm[0][1]) if (best_cm[0][0] + best_cm[0][1]) > 0 else 0
    
    print(f"\nMetrics from Confusion Matrix (Fake class):")
    print(f"  Accuracy: {accuracy:.4f}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall: {recall:.4f}")
else:
    print("Training history or confusion matrix not found. Please run the training cell first.")


In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import os

# Define the path to the best model checkpoint
out_dir = f'{run_base_dir}/xception_dfdc'  # Make sure this matches your training output directory
best_model_path = os.path.join(out_dir, 'xception_best.pt')

# Define the path to the test data
data_dir = crops_base_dir # Make sure this matches your crops directory
test_data_dir = os.path.join(data_dir, 'test')

# Check if the best model checkpoint exists
if not os.path.exists(best_model_path):
    print(f"Error: Best model checkpoint not found at {best_model_path}")
else:
    # Load the best model state
    checkpoint = torch.load(best_model_path)

    # Build the model architecture (needs to match the one used for training)
    import timm
    model = timm.create_model("xception", pretrained=False, num_classes=2) # pretrained=False because we are loading a trained model
    model.load_state_dict(checkpoint['model_state'])

    # Set the model to evaluation mode
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)
    model.eval()
    print(f"Loaded best model from {best_model_path} and set to evaluation mode on {device}.")

    # Define the transformations for the test set (same as validation)
    img_size = 299
    test_tf = transforms.Compose([
        transforms.Resize(img_size),
        transforms.CenterCrop(img_size),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])

    # Create the test dataset and dataloader
    try:
        test_ds = datasets.ImageFolder(test_data_dir, transform=test_tf)
        test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=8, pin_memory=True) # Use a reasonable batch size
        print(f"Created test dataset with {len(test_ds)} images.")

        # Evaluate the model on the test set
        print("Evaluating model on the test set...")
        test_metrics, test_cm = evaluate_frame_level(model, test_loader, device)

        print("\nTest Set Metrics:")
        print(f"  Accuracy: {test_metrics['acc']:.4f}")
        print(f"  F1 Score: {test_metrics['f1']:.4f}")
        print(f"  AUC: {test_metrics['auc']:.4f}")
        print(f"  AP (Fake): {test_metrics['ap_fake']:.4f}")
        
        print("\nTest Set Confusion Matrix:")
        print(test_cm)

    except FileNotFoundError:
        print(f"Error: Test data directory not found at {test_data_dir}. Please ensure the path is correct.")
    except Exception as e:
        print(f"An error occurred during test set evaluation: {e}")

Loaded best model from /content/drive/MyDrive/DeepfakeData/CompleteData/organized/runs/xception_dfdc/xception_best.pt and set to evaluation mode on cuda.
Created test dataset with 4004 images.
Evaluating model on the test set...

Test Set Metrics:
  Accuracy: 0.9795
  F1 Score: 0.8423
  AUC: 0.9492
  AP (Fake): 0.8786


In [ ]:
# Visualize Test Set Confusion Matrix
import matplotlib.pyplot as plt
import seaborn as sns

if 'test_cm' in locals():
    plt.figure(figsize=(8, 6))
    # Note: Classes are ['fake_videos', 'real_videos'], so index 0=Fake, index 1=Real
    sns.heatmap(test_cm, annot=True, fmt='d', cmap='Greens', 
                xticklabels=['Fake', 'Real'], 
                yticklabels=['Fake', 'Real'],
                cbar_kws={'label': 'Count'})
    plt.title('Confusion Matrix - Test Set', fontsize=14, fontweight='bold')
    plt.xlabel('Predicted Label', fontsize=12)
    plt.ylabel('True Label', fontsize=12)
    plt.tight_layout()
    plt.show()
    
    # Print confusion matrix details
    # test_cm structure: [[Fake->Fake, Fake->Real], [Real->Fake, Real->Real]]
    print("Test Set Confusion Matrix Details:")
    print(f"  True Positives (Fake->Fake): {test_cm[0][0]}")
    print(f"  False Negatives (Fake->Real): {test_cm[0][1]}")
    print(f"  False Positives (Real->Fake): {test_cm[1][0]}")
    print(f"  True Negatives (Real->Real): {test_cm[1][1]}")
else:
    print("Test confusion matrix not found. Please run the test evaluation cell first.")
